# Thumbnail DINO To Embedding Atlas

This notebook takes a thumbnail config exported from `thumbnail_explorer_app.py`, rebuilds the exact same thumbnail dataset, computes frozen DINOv2 features, saves thumbnail PNGs plus metadata, computes a 2D projection and nearest neighbors, and prepares a dataframe for `EmbeddingAtlasWidget`.

The projection is computed from image embeddings, not from the text description.

In [19]:
import sys
from pathlib import Path

sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import os

from movebench.thumbnail_config import load_thumbnail_generation_config, thumbnail_config_run_slug
from movebench.thumbnail_atlas import (
    
    build_atlas_dataframe,
    build_thumbnail_metadata,
    save_atlas_artifacts,
    save_thumbnail_pngs,
    serialize_neighbors,
)
from movebench.thumbnail_embedding import compute_neighbors, compute_thumbnail_embeddings, project_embeddings

In [30]:
# Point this to a config exported from the Streamlit thumbnail explorer.
CONFIG_PATH = Path('/data/vision/beery/scratch/shir/codes/movebench/configs/thumbnails/Carter_28708a9c_raw_merged__all__thumbnail_config.json')

# Output root for thumbnails, metadata, and embeddings.
OUTPUT_ROOT = Path('../notebooks_outputs/thumbnail_atlas/',os.path.basename(CONFIG_PATH).split('.')[0])

EMBEDDING_CONFIG = {
    'model_name': 'dinov2_vits14',
    'device_preference': 'cuda',
    'batch_size': 32,
    'image_size': 224,
}

PROJECTION_CONFIG = {
    'method': 'umap',
    'random_state': 17,
    'n_neighbors': 20,
    'min_dist': 0.10,
    'perplexity': 30.0,
}

In [31]:
config = load_thumbnail_generation_config(CONFIG_PATH)
run_slug = thumbnail_config_run_slug(config)
run_dir = OUTPUT_ROOT / run_slug
run_dir.mkdir(parents=True, exist_ok=True)

config, build, metadata = build_thumbnail_metadata(config)
print(f'Loaded config for {config["data"]["csv_path"]}')
print(f'Thumbnails: {len(build.images):,}')
print(f'Individuals: {metadata["individual_id"].astype(str).nunique():,}')
metadata.head()

Loaded config for /data/vision/beery/scratch/shir/codes/vibecleaning/data/movement_example/cleaned_topout_v1/Carter_28708a9c_raw_merged.csv
Thumbnails: 253
Individuals: 6


,global_chunk_idx,individual_id,chunk_idx,split,start_time,end_time,n_fixes,start_lon,start_lat,end_lon,...,n_outlier_fixes,has_any_outlier,outlier_frac,clean,source_name,csv_path,selected_split,overlap_state,thumbnail_id,description
0,0,Ada,0,train,2016-07-31 14:00:37,2016-08-03 23:01:05,50,141.044717,-33.134447,141.043308,...,2,True,0.04,False,Carter_28708a9c_raw_merged,/data/vision/beery/scratch/shir/codes/vibeclea...,all,algorithm only,Carter_28708a9c_raw_merged|train|Ada|0|2016-07...,source=Carter_28708a9c_raw_merged split=train ...
1,1,Ada,1,train,2016-08-04 00:36:44,2016-08-06 15:00:36,50,141.043261,-33.136456,141.058403,...,1,True,0.02,False,Carter_28708a9c_raw_merged,/data/vision/beery/scratch/shir/codes/vibeclea...,all,algorithm only,Carter_28708a9c_raw_merged|train|Ada|1|2016-08...,source=Carter_28708a9c_raw_merged split=train ...
2,2,Ada,2,train,2016-08-06 16:00:36,2016-08-09 12:00:36,50,141.060603,-33.130097,141.057767,...,3,True,0.06,False,Carter_28708a9c_raw_merged,/data/vision/beery/scratch/shir/codes/vibeclea...,all,algorithm only,Carter_28708a9c_raw_merged|train|Ada|2|2016-08...,source=Carter_28708a9c_raw_merged split=train ...
3,3,Ada,3,train,2016-08-09 13:00:38,2016-08-11 20:01:06,50,141.060511,-33.129531,141.041592,...,0,False,0.00,True,Carter_28708a9c_raw_merged,/data/vision/beery/scratch/shir/codes/vibeclea...,all,clean,Carter_28708a9c_raw_merged|train|Ada|3|2016-08...,source=Carter_28708a9c_raw_merged split=train ...
4,4,Ada,4,train,2016-08-11 21:00:37,2016-08-14 07:00:44,50,141.041561,-33.134989,141.046761,...,2,True,0.04,False,Carter_28708a9c_raw_merged,/data/vision/beery/scratch/shir/codes/vibeclea...,all,algorithm only,Carter_28708a9c_raw_merged|train|Ada|4|2016-08...,source=Carter_28708a9c_raw_merged split=train ...


In [32]:
metadata.has_any_outlier.sum()

108

In [33]:
thumbnail_paths = save_thumbnail_pngs(build.images, metadata, out_dir=run_dir / 'thumbnails')
metadata = metadata.copy()
metadata['thumbnail_path'] = thumbnail_paths
metadata[['thumbnail_id', 'thumbnail_path', 'description']].head()

,thumbnail_id,thumbnail_path,description
0,Carter_28708a9c_raw_merged|train|Ada|0|2016-07...,/data/vision/beery/scratch/shir/codes/movebenc...,source=Carter_28708a9c_raw_merged split=train ...
1,Carter_28708a9c_raw_merged|train|Ada|1|2016-08...,/data/vision/beery/scratch/shir/codes/movebenc...,source=Carter_28708a9c_raw_merged split=train ...
2,Carter_28708a9c_raw_merged|train|Ada|2|2016-08...,/data/vision/beery/scratch/shir/codes/movebenc...,source=Carter_28708a9c_raw_merged split=train ...
3,Carter_28708a9c_raw_merged|train|Ada|3|2016-08...,/data/vision/beery/scratch/shir/codes/movebenc...,source=Carter_28708a9c_raw_merged split=train ...
4,Carter_28708a9c_raw_merged|train|Ada|4|2016-08...,/data/vision/beery/scratch/shir/codes/movebenc...,source=Carter_28708a9c_raw_merged split=train ...


In [34]:
import torch
torch.cuda.is_available()

True

In [35]:
embeddings, embedding_info = compute_thumbnail_embeddings(build.images, embedding_config=EMBEDDING_CONFIG)
projection, projection_info, projection_warnings = project_embeddings(
    embeddings,
    projection_config=PROJECTION_CONFIG,
    cache_root=run_dir / '.cache',
)
neighbor_idx, neighbor_dist = compute_neighbors(embeddings)
neighbors = serialize_neighbors(
    neighbor_idx,
    metadata['thumbnail_id'].astype(str).tolist(),
    neighbor_distances=neighbor_dist,
)

print('Embedding info:', embedding_info)
print('Projection info:', projection_info)
if projection_warnings:
    print('Projection warnings:')
    for warning in projection_warnings:
        print('-', warning)
embeddings.shape, projection.shape

/data/vision/beery/scratch/shir/micromamba/envs/movebench-torch/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Embedding info: {'device': 'cuda', 'model_name': 'dinov2_vits14', 'image_size': 224, 'batch_size': 32}
Projection info: {'method': 'umap'}


((253, 384), (253, 2))

In [36]:
atlas_df = build_atlas_dataframe(
    metadata,
    projection=projection,
    neighbors=neighbors,
    thumbnail_paths=thumbnail_paths,
)

save_atlas_artifacts(
    config=config,
    atlas_df=atlas_df,
    embeddings=embeddings,
    out_root=OUTPUT_ROOT,
)

atlas_df[['thumbnail_id', 'individual_id', 'split', 'overlap_state', 'projection_x', 'projection_y', 'neighbors']].head()

,thumbnail_id,individual_id,split,overlap_state,projection_x,projection_y,neighbors
0,Carter_28708a9c_raw_merged|train|Ada|0|2016-07...,Ada,train,algorithm only,10.500845,12.138247,{'ids': ['Carter_28708a9c_raw_merged|test|Bea|...
1,Carter_28708a9c_raw_merged|train|Ada|1|2016-08...,Ada,train,algorithm only,10.531215,13.511557,{'ids': ['Carter_28708a9c_raw_merged|test|Don|...
2,Carter_28708a9c_raw_merged|train|Ada|2|2016-08...,Ada,train,algorithm only,10.696797,13.330194,{'ids': ['Carter_28708a9c_raw_merged|train|Ada...
3,Carter_28708a9c_raw_merged|train|Ada|3|2016-08...,Ada,train,clean,10.497101,13.066350,{'ids': ['Carter_28708a9c_raw_merged|train|Ada...
4,Carter_28708a9c_raw_merged|train|Ada|4|2016-08...,Ada,train,algorithm only,11.389180,13.353161,{'ids': ['Carter_28708a9c_raw_merged|test|Bea|...


In [37]:
from embedding_atlas.widget import EmbeddingAtlasWidget

widget = EmbeddingAtlasWidget(
    atlas_df,
    text='description',
    x='projection_x',
    y='projection_y',
    neighbors='neighbors',
)
widget